# Evaluation of EC3D Dataset using Heuristic Rules
This notebook loads 3D coordinates from the EC3D dataset, converts them into angle/distance metrics, evaluates them against heuristic thresholds from the `Code/` directory to measure Top-1 Error Accuracy and FPS execution speed.

In [1]:
import pickle
import numpy as np
import sys
import os
import time

# Add Code to path to import process_video and evaluation logically
sys.path.append(os.path.abspath('../../Code'))
from process_video import get_landmark_coords_from_fit3d
from evaluation import evaluate_metrics
from exercise_config import EXERCISE_TO_CONFIG, JOINT_DEFINITIONS

def compute_metrics(pose, metric_configs):
    frame_metrics = {}
    for metric_name, config in metric_configs.items():
        joints = config.get('joints', [])
        metric_type = config.get('type', 'angle')
        for side in ['LEFT', 'RIGHT']:
            points = []
            valid = True
            for j in joints:
                p = get_landmark_coords_from_fit3d(pose, j, side)
                if p is None:
                    valid = False; break
                points.append(p)
            if not valid: continue
            
            value = None
            if metric_type == 'angle' and len(points) == 3:
                a, b, c = points[0], points[1], points[2]
                if isinstance(a, str): a = np.array([b[0], b[1] - 0.5, b[2]]) if a == 'vertical' else np.array([b[0] + 0.5, b[1], b[2]])
                if isinstance(c, str): c = np.array([b[0], b[1] - 0.5, b[2]]) if c == 'vertical' else np.array([b[0] + 0.5, b[1], b[2]])
                ba, bc = a - b, c - b
                n1, n2 = np.linalg.norm(ba), np.linalg.norm(bc)
                if n1 > 1e-6 and n2 > 1e-6:
                    value = np.degrees(np.arccos(np.clip(np.dot(ba, bc) / (n1 * n2), -1.0, 1.0)))
            elif metric_type == 'horizontal_distance' and len(points) >= 2:
                value = np.abs(points[0][0] - points[1][0]) / 0.5
            elif metric_type == 'vertical_distance' and len(points) >= 2:
                value = np.abs(points[0][1] - points[1][1]) / 0.5
            elif metric_type == 'distance_from_line' and len(points) == 3:
                p0, p1, p2 = points[0], points[1], points[2]
                value = np.linalg.norm(np.cross(p1 - p0, p0 - p2)) / max(np.linalg.norm(p1 - p0), 1e-6)
            
            if value is not None:
                frame_metrics[f"{metric_name}_{side.lower()}"] = value
    return frame_metrics


print("Loading Dataset...")
try:
    with open('../../Official Dataset/EC3D_dataset/EC3D_dataset/data_3D.pickle', 'rb') as f:
        data = pickle.load(f)
except FileNotFoundError:
    print("Please run this notebook from its directory or update paths.")
    data = None

if data is not None:
    labels = data['labels'] # [act, sub, lab, rep, frame]
    poses = data['poses'] # (N, 3, 25) 
    poses = np.transpose(poses, (0, 2, 1)) # (N, 25, 3)

    EXERCISE_MAP = {
        'SQUAT': 'squat',
        'Lunges': 'lunge', 
        'plank': 'plank'
    }

    # For EC3D mapping, group by lowercased names
    activities_lower = np.char.lower(labels[:, 0].astype(str))

    for tgt_data, tgt_conf in EXERCISE_MAP.items():
        if tgt_conf not in EXERCISE_TO_CONFIG:
            for k in EXERCISE_TO_CONFIG.keys():
                if tgt_conf in k:
                    tgt_conf = k
                    break
            
            if tgt_conf not in EXERCISE_TO_CONFIG:
                print(f"\nSkipping {tgt_data}, config not found. ( Lunges aren't in exercise_config.py! )")
                continue

        config = EXERCISE_TO_CONFIG[tgt_conf]
        metric_keys = config['metrics']
        
        metric_configs = {m: JOINT_DEFINITIONS.get(m) for m in metric_keys if m in JOINT_DEFINITIONS}
        ref_thresholds = config['thresholds']

        idx = np.where(activities_lower == tgt_data.lower())[0]
        if len(idx) == 0:
            print(f"Skipping {tgt_data}, no labels found in EC3D set")
            continue

        tgt_labels = labels[idx]
        tgt_poses = poses[idx]

        sequences = {}
        for i, (act, sub, lab, rep, frame) in enumerate(tgt_labels):
            seq_id = f"{sub}_{rep}_{lab}"
            if seq_id not in sequences:
                sequences[seq_id] = {'lab': str(lab), 'frames': []}
            sequences[seq_id]['frames'].append(tgt_poses[i])

        correct_predict = 0
        total_predict = len(sequences)
        correct_lab1_count = 0 
        total_lab1_count = 0
        correct_lab_mistake_count = 0
        total_lab_mistake_count = 0
        
        print(f"\n====================================")
        print(f"Evaluating {tgt_data.upper()} ({len(sequences)} sequences)")

        total_frames_processed = 0
        start_time = time.time()
        
        for seq_id, seq_data in sequences.items():
            lab = seq_data['lab']
            is_gt_correct = (lab == '1')
            error_frames = 0
            
            for pose in seq_data['frames']:
                frame_metrics = compute_metrics(pose, metric_configs)
                state = {'feedbacks': {}, 'frame_count': 0}
                is_curr_good, msgs, _ = evaluate_metrics(frame_metrics, metric_configs, ref_thresholds, state, active_sides=['left', 'right'])
                if not is_curr_good:
                    error_frames += 1
                total_frames_processed += 1
            
            threshold_frames = max(1, int(len(seq_data['frames']) * 0.10))
            pred_is_correct = (error_frames <= threshold_frames)
            
            if is_gt_correct:
                total_lab1_count += 1
                if pred_is_correct: correct_lab1_count += 1
            else:
                total_lab_mistake_count += 1
                if not pred_is_correct: correct_lab_mistake_count += 1
                
            if pred_is_correct == is_gt_correct:
                correct_predict += 1

        end_time = time.time()
        duration = end_time - start_time
        fps = total_frames_processed / duration if duration > 0 else 0

        overall_acc = correct_predict / total_predict if total_predict > 0 else 0
        acc_correct_form = correct_lab1_count / total_lab1_count if total_lab1_count > 0 else 0
        acc_mistake_form = correct_lab_mistake_count / total_lab_mistake_count if total_lab_mistake_count > 0 else 0
        
        print(f"Overall Accuracy: {overall_acc:.2%}")
        print(f"Accuracy on Correct Form (lab=1): {acc_correct_form:.2%}")
        print(f"Accuracy on Mistakes (lab!=1): {acc_mistake_form:.2%}")
        print(f"Total Frames Processed: {total_frames_processed}")
        print(f"Latency/FPS (Heuristic Execution): {fps:.2f} Frames Per Second")


Loading Dataset...

Evaluating SQUAT (141 sequences)
FAILED HIGH: BACK_ANGLE_VERTICAL left val 105.62601029835943 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 107.60675646212808 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 109.48696001175514 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 111.26006923526343 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 112.91865889644636 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 114.45937050702602 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 115.84962004547951 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 117.03466087580257 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 117.95537029016745 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 118.5489074433304 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 118.84707060627998 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 118.90487085604767 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 118.82516815303258 > 105.0
FAILED HIGH: BACK_ANGLE_VERTICAL left val 118.74898